# Architecture Sweep — Raw Trajectory DL

Systematically varies CNN and LSTM capacity (small / medium / large) on the
raw step-vector input.  Medium matches the baseline in `run_deep_learning_experiments.py`.

**Same protocol throughout:**
- Nucleus-level 80/20 split (GroupShuffleSplit, `random_state=42`)
- 5-fold CV on training set, then one final model on full training set
- AdamW lr=1e-3 · cosine annealing 150 epochs · early stopping patience=20

| Variant | CNN channels | LSTM hidden |
|---------|-------------|-------------|
| Small   | 32 → 64     | 32          |
| Medium  | 64 → 128    | 64          |
| Large   | 128 → 256   | 128         |

---
**Running on Google Colab?**  
1. Upload your data folder to Google Drive (keep the same folder structure as on the lab machine).  
2. Run the *Environment setup* cell — it will mount Drive automatically.  
3. Set `DRIVE_BASE` in the *Paths & config* cell to point to your Drive folder.

## Environment setup
Detects Colab vs local, mounts Drive, installs any missing packages.

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')

# All packages below are pre-installed in Colab / standard conda envs.
# Uncomment if anything is missing:
# !pip install -q torch numpy pandas scikit-learn

## Paths & config

**Edit `DRIVE_BASE` if running on Colab** — set it to wherever you uploaded
the `data for modeling` folder inside your Drive.
Leave `LOCAL_BASE` alone; it is used automatically when not in Colab.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, KFold, GroupShuffleSplit
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# ── Path configuration ─────────────────────────────────────────────────────
import os
# Local: set DATA_ROOT to override the committed modeling data.
_LOCAL_DATA_CANDIDATES = [Path('../data'), Path('data'), Path('trajectory_to_nuclear_features/data')]
LOCAL_BASE = (Path(os.environ['DATA_ROOT']) if 'DATA_ROOT' in os.environ
              else next((p for p in _LOCAL_DATA_CANDIDATES if p.exists()), _LOCAL_DATA_CANDIDATES[0]))

# Colab / Drive — edit the path after MyDrive/ to match where you put the data
DRIVE_BASE = Path('/content/drive/MyDrive/CS273B/data for modeling')

BASE = DRIVE_BASE if IN_COLAB else LOCAL_BASE
print('Data root:', BASE)
assert BASE.exists(), f'Path not found: {BASE}'

LOCUS_CSV = BASE / 'chr3/locus_feature_table.csv'
NUC_CSV   = BASE / 'chr3/nucleus_feature_table.csv'
B1_MAP    = BASE / 'trajectories/batch1/Nuc_number_mapping.csv'
C3_MAP    = BASE / 'trajectories/chr3_batch/Nuc_number_mapping.csv'
B1_DIR    = BASE / 'trajectories/batch1'
C3_DIR    = BASE / 'trajectories/chr3_batch'
OUT_DIR   = BASE / 'outputs/deep_learning'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

# ── Targets ────────────────────────────────────────────────────────────────
TARGETS = [
    'area_um2',
    'local_intensity_mean',
    'local_to_nuc_ratio',
    'nuc_intensity_mean',
    'dist_to_membrane_nm',
    'dist_to_centroid_nm',
    'norm_radial_pos',
]
LEAKAGE_TARGETS = ['x_nm_mean', 'y_nm_mean']

# ── Hyperparameters ─────────────────────────────────────────────────────────
T_MAX     = 29
MIN_STEPS = 3
N_FEAT    = 2
SEED      = 42
BATCH_SZ  = 32
N_EPOCHS  = 150
LR        = 1e-3
WD        = 1e-4
PATIENCE  = 20
N_FOLDS   = 5

torch.manual_seed(SEED)
np.random.seed(SEED)

## Data loading helpers
Identical to `run_deep_learning_experiments.py`.

In [ ]:
def build_chr3_canonical_map(c3_map_path):
    df = pd.read_csv(c3_map_path)
    df['bare_subdir'] = df['source_subsub_directory'].str.split('/').str[-1]
    return (df.groupby(['bare_subdir', 'original_filename'])['Nuc_num']
              .min().to_dict())


def build_lookup(map_path, traj_dir, chr3_canon=None):
    df = pd.read_csv(map_path)
    lookup = {}
    for _, row in df.iterrows():
        nid  = row['source_subsub_directory']
        orig = row['original_filename']
        nnum = row['Nuc_num']
        bare = nid.split('/')[-1]
        if chr3_canon is not None:
            canon = chr3_canon.get((bare, orig))
            if canon is not None and nnum != canon:
                continue
        locus_id = orig.replace('_traj_m2DGaussian_cleaned.csv', '')
        fpath = traj_dir / f'Nuc{nnum}_{orig}'
        if fpath.exists():
            lookup[(nid, locus_id)] = fpath
    return lookup


def traj_to_steps(df_traj):
    df     = df_traj.sort_values('frame').reset_index(drop=True)
    frames = df['frame'].values.astype(int)
    x      = df['x_nm'].values.astype(float)
    y      = df['y_nm'].values.astype(float)
    consec = np.diff(frames) == 1
    dx     = np.diff(x)[consec]
    dy     = np.diff(y)[consec]
    n      = len(dx)
    if n < MIN_STEPS:
        return None, None
    steps = np.stack([dx, dy], axis=1).astype(np.float32)
    if n > T_MAX:
        steps, n = steps[-T_MAX:], T_MAX
    padded = np.zeros((T_MAX, N_FEAT), dtype=np.float32)
    mask   = np.zeros(T_MAX, dtype=bool)
    padded[:n] = steps
    mask[:n]   = True
    return padded, mask


def build_dataset(lookup, locus_df, nuc_df):
    g = locus_df[locus_df['locus_id'].str.startswith('G')].copy()
    traj_agg = (g.groupby(['nucleus_id', 'locus_id'])
                  .agg(n_frames             = ('frame',                'count'),
                       local_intensity_mean = ('local_intensity_mean', 'median'),
                       local_to_nuc_ratio   = ('local_to_nuc_ratio',   'median'),
                       dist_to_membrane_nm  = ('dist_to_membrane_nm',  'median'),
                       dist_to_centroid_nm  = ('dist_to_centroid_nm',  'median'),
                       norm_radial_pos      = ('norm_radial_pos',      'median'),
                       x_nm_mean            = ('x_nm',                 'mean'),
                       y_nm_mean            = ('y_nm',                 'mean'))
                  .reset_index())
    traj_agg = traj_agg[traj_agg['n_frames'] >= 5]
    nuc_agg  = (nuc_df.groupby('nucleus_id')[['area_um2', 'nuc_intensity_mean']]
                      .median().reset_index())
    traj_agg = traj_agg.merge(nuc_agg, on='nucleus_id', how='left')

    X_list, mask_list, Y_list, YL_list, nid_list = [], [], [], [], []
    skip_no_traj = skip_short = 0
    for _, row in traj_agg.iterrows():
        key = (row['nucleus_id'], row['locus_id'])
        if key not in lookup:
            skip_no_traj += 1; continue
        try:
            df_traj = pd.read_csv(lookup[key])
        except Exception:
            skip_no_traj += 1; continue
        steps, mask = traj_to_steps(df_traj)
        if steps is None:
            skip_short += 1; continue
        X_list.append(steps)
        mask_list.append(mask)
        Y_list.append([
            row['area_um2'], row['local_intensity_mean'],
            row['local_to_nuc_ratio'], row['nuc_intensity_mean'],
            row['dist_to_membrane_nm'], row['dist_to_centroid_nm'],
            row['norm_radial_pos'],
        ])
        YL_list.append([row['x_nm_mean'], row['y_nm_mean']])
        nid_list.append(row['nucleus_id'])
    print(f'  Skipped (no CSV): {skip_no_traj}  |  too short: {skip_short}')
    return (np.stack(X_list).astype(np.float32),
            np.stack(mask_list),
            np.array(Y_list, dtype=np.float32),
            np.array(YL_list, dtype=np.float32),
            np.array(nid_list))

## Model definitions

In [ ]:
class TrajCNN(nn.Module):
    """1D CNN with parameterised base channel width."""
    def __init__(self, n_targets, n_ch=64, drop=0.3):
        super().__init__()
        self.n_ch = n_ch
        self.conv = nn.Sequential(
            nn.Conv1d(N_FEAT, n_ch,     kernel_size=3, padding=1),
            nn.BatchNorm1d(n_ch), nn.ReLU(), nn.Dropout(drop),
            nn.Conv1d(n_ch,   n_ch * 2, kernel_size=3, padding=1),
            nn.BatchNorm1d(n_ch * 2), nn.ReLU(), nn.Dropout(drop),
        )
        self.head = nn.Sequential(
            nn.Linear(n_ch * 2, 64), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(64, n_targets),
        )

    def forward(self, x, mask):
        x = self.conv(x.permute(0, 2, 1)).permute(0, 2, 1)
        m = mask.unsqueeze(-1).float()
        return self.head((x * m).sum(1) / m.sum(1).clamp(min=1))

    def param_count(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class TrajLSTM(nn.Module):
    """Bidirectional LSTM with parameterised hidden size."""
    def __init__(self, n_targets, hidden=64, drop=0.3):
        super().__init__()
        self.hidden = hidden
        self.lstm = nn.LSTM(N_FEAT, hidden, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=drop)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(64, n_targets),
        )

    def forward(self, x, mask):
        lengths = mask.sum(dim=1).cpu().clamp(min=1)
        packed  = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        return self.head(torch.cat([h[-2], h[-1]], dim=-1))

    def param_count(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Quick param count check
for n_ch in [32, 64, 128]:
    m = TrajCNN(len(TARGETS), n_ch=n_ch)
    print(f'CNN n_ch={n_ch:3d} → {m.param_count():,} params')
for h in [32, 64, 128]:
    m = TrajLSTM(len(TARGETS), hidden=h)
    print(f'LSTM hidden={h:3d} → {m.param_count():,} params')

## Training helpers

In [ ]:
class TrajDataset(Dataset):
    def __init__(self, X, masks, Y):
        self.X     = torch.tensor(X,     dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.bool)
        self.Y     = torch.tensor(Y,     dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.masks[i], self.Y[i]


def masked_mse(pred, target):
    valid = ~torch.isnan(target)
    loss  = torch.tensor(0.0, device=pred.device)
    for t in range(target.shape[1]):
        m = valid[:, t]
        if m.sum() > 0:
            loss = loss + F.mse_loss(pred[m, t], target[m, t])
    return loss


def fit_model(model_factory, X_tr, m_tr, Y_tr, X_va, m_va, Y_va):
    model = model_factory().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
    tr_ld = DataLoader(TrajDataset(X_tr, m_tr, Y_tr), BATCH_SZ, shuffle=True)
    va_ld = DataLoader(TrajDataset(X_va, m_va, Y_va), BATCH_SZ)
    best_val, best_state, wait = float('inf'), None, 0
    for _ in range(N_EPOCHS):
        model.train()
        for Xb, mb, Yb in tr_ld:
            opt.zero_grad()
            masked_mse(model(Xb.to(DEVICE), mb.to(DEVICE)),
                       Yb.to(DEVICE)).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        val = 0.0
        with torch.no_grad():
            for Xb, mb, Yb in va_ld:
                val += masked_mse(model(Xb.to(DEVICE), mb.to(DEVICE)),
                                  Yb.to(DEVICE)).item()
        val /= len(va_ld)
        if val < best_val - 1e-6:
            best_val   = val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict(model, X, masks, Y):
    model.eval()
    out = []
    for Xb, mb, _ in DataLoader(TrajDataset(X, masks, Y), BATCH_SZ):
        out.append(model(Xb.to(DEVICE), mb.to(DEVICE)).cpu())
    return torch.cat(out).numpy()


def norm_X(X_tr, X_te):
    mu  = X_tr.mean(axis=(0, 1), keepdims=True)
    std = X_tr.std(axis=(0, 1),  keepdims=True) + 1e-8
    return (X_tr - mu) / std, (X_te - mu) / std

def norm_Y(Y_tr, Y_te):
    mu  = np.nanmean(Y_tr, axis=0)
    std = np.nanstd(Y_tr,  axis=0) + 1e-8
    return (Y_tr - mu) / std, (Y_te - mu) / std, mu, std

def r2(yt, yp):
    m = ~np.isnan(yt)
    return r2_score(yt[m], yp[m]) if m.sum() >= 2 else np.nan

def pearson(yt, yp):
    m = ~np.isnan(yt)
    return float(np.corrcoef(yt[m], yp[m])[0, 1]) if m.sum() >= 2 else np.nan

## Load dataset

In [ ]:
print('Building lookup tables ...')
c3_canon  = build_chr3_canonical_map(C3_MAP)
lookup_b1 = build_lookup(B1_MAP, B1_DIR)
lookup_c3 = build_lookup(C3_MAP, C3_DIR, chr3_canon=c3_canon)
lookup    = {**lookup_b1, **lookup_c3}
print(f'  Batch 1: {len(lookup_b1)} | Chr3 (deduped): {len(lookup_c3)} | Combined: {len(lookup)}')

print('\nLoading feature tables ...')
locus_df = pd.read_csv(LOCUS_CSV)
nuc_df   = pd.read_csv(NUC_CSV)

print('\nBuilding dataset ...')
X, masks, Y, Y_leak, nucleus_ids = build_dataset(lookup, locus_df, nuc_df)
print(f'Dataset: {len(X)} trajectories, {len(set(nucleus_ids))} unique nuclei')
print(f'Steps: min={masks.sum(1).min()}  median={np.median(masks.sum(1)):.0f}  max={masks.sum(1).max()}')

## Nucleus-level train / test split
Computed **once** and shared across all variants — every variant sees the same test set.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx, te_idx = next(gss.split(X, Y, groups=nucleus_ids))

tr_nucs = set(nucleus_ids[tr_idx])
te_nucs = set(nucleus_ids[te_idx])
overlap  = tr_nucs & te_nucs
print(f'Train: {len(tr_idx)} trajectories, {len(tr_nucs)} unique nuclei')
print(f'Test:  {len(te_idx)} trajectories, {len(te_nucs)} unique nuclei')
print(f'Nucleus overlap: {len(overlap)} ({"✓ none" if not overlap else "✗ OVERLAP"})')

X_tr_r, m_tr = X[tr_idx],   masks[tr_idx]
X_te_r, m_te = X[te_idx],   masks[te_idx]
Y_tr_r        = Y[tr_idx];   Y_te_r = Y[te_idx]
YL_tr         = Y_leak[tr_idx]; YL_te = Y_leak[te_idx]

X_tr_n, X_te_n             = norm_X(X_tr_r, X_te_r)
Y_tr_n, Y_te_n, y_mu, y_std = norm_Y(Y_tr_r, Y_te_r)

## Architecture sweep

6 variants × (5 folds + 1 final) = 36 training runs.  
On Colab T4 GPU this takes ~15–20 min total.

In [ ]:
VARIANTS = [
    ('CNN-small',   TrajCNN,  {'n_ch': 32}),
    ('CNN-medium',  TrajCNN,  {'n_ch': 64}),    # baseline
    ('CNN-large',   TrajCNN,  {'n_ch': 128}),
    ('LSTM-small',  TrajLSTM, {'hidden': 32}),
    ('LSTM-medium', TrajLSTM, {'hidden': 64}),   # baseline
    ('LSTM-large',  TrajLSTM, {'hidden': 128}),
]

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fi_final, vi_final = train_test_split(
    np.arange(len(X_tr_n)), test_size=0.1, random_state=SEED)

sweep_results = []

for label, ModelCls, kwargs in VARIANTS:
    factory   = lambda cls=ModelCls, kw=kwargs: cls(len(TARGETS), **kw)
    param_cnt = factory().param_count()
    print(f'\n{"─"*60}')
    print(f'{label}  ({param_cnt:,} params)')
    print(f'{"─"*60}')

    cv_preds = np.full_like(Y_tr_n, np.nan)
    for fold, (fi, vi) in enumerate(kf.split(X_tr_n)):
        print(f'  Fold {fold+1}/{N_FOLDS}...', end=' ', flush=True)
        m = fit_model(factory,
                      X_tr_n[fi], m_tr[fi], Y_tr_n[fi],
                      X_tr_n[vi], m_tr[vi], Y_tr_n[vi])
        cv_preds[vi] = predict(m, X_tr_n[vi], m_tr[vi], Y_tr_n[vi])
        print('done')

    print('  Final model...', end=' ', flush=True)
    final = fit_model(factory,
                      X_tr_n[fi_final], m_tr[fi_final], Y_tr_n[fi_final],
                      X_tr_n[vi_final], m_tr[vi_final], Y_tr_n[vi_final])
    print('done')

    te_preds = predict(final, X_te_n, m_te, Y_te_n) * y_std + y_mu

    for i, t in enumerate(TARGETS):
        cv_r2_ = r2(Y_tr_n[:, i], cv_preds[:, i])
        te_r2_ = r2(Y_te_r[:, i],  te_preds[:, i])
        te_cor = pearson(Y_te_r[:, i], te_preds[:, i])
        print(f'  {t:<24} CV R²={cv_r2_:>6.3f}  Test R²={te_r2_:>6.3f}  corr={te_cor:>6.3f}')
        sweep_results.append({
            'variant': label, 'target': t, 'params': param_cnt,
            'cv_r2': cv_r2_, 'test_r2': te_r2_, 'test_corr': te_cor,
        })

print('\nSweep complete.')

## Results summary

In [ ]:
results_df = pd.DataFrame(sweep_results)

pivot = results_df.pivot_table(
    index='target', columns='variant', values='test_r2', aggfunc='first'
)
col_order = [v[0] for v in VARIANTS]
pivot = pivot[[c for c in col_order if c in pivot.columns]]

print('Test R² by variant and target')
print('=' * 80)
print(pivot.round(3).to_string())

print('\nBest variant per target (test R²):')
best = results_df.loc[results_df.groupby('target')['test_r2'].idxmax()]
print(best[['target', 'variant', 'test_r2', 'test_corr']].to_string(index=False))

In [ ]:
out_path = OUT_DIR / 'dl_arch_sweep_results.csv'
results_df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

# If on Colab, also save to the current working directory so you can download it
if IN_COLAB:
    results_df.to_csv('dl_arch_sweep_results.csv', index=False)
    from google.colab import files
    files.download('dl_arch_sweep_results.csv')